# M2 — Segmentación de vehículos para el sistema de peaje
**Google Colab · Ultralytics YOLOv8n-seg · Roboflow v1**

Este notebook entrena únicamente M2 (`auto`, `bus`, `camion`, `minibus`, `moto`). M1 (OCR) y M3 (ejes) quedan fuera de este entrenamiento.

## Antes de ejecutar
1. Guarda una copia de este notebook en Drive. En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** (T4 si está disponible).
2. En Roboflow verifica que la versión 1 de tu proyecto incluya las cinco clases y las imágenes revisadas. Si generaste otra versión cambia `VERSION` en la configuración.
3. En Colab abre **Secretos** (icono de llave), crea `ROBOFLOW_API_KEY` con tu clave privada vigente y activa acceso al notebook. Si prefieres usar un ZIP, puedes cambiar `SOURCE` a `zip` y subirlo a la ruta indicada. No pongas claves en el código ni compartas el dataset si contiene información identificable. Revoca la clave anteriormente compartida y usa una nueva.
4. Ejecuta las celdas en orden. No uses Train Model en Roboflow. El entrenamiento se hace en Colab y guarda los resultados directamente en Drive.

**Resultado principal:** `Mi unidad/Sistema-Peaje/modelos/vehiculos.pt`. Los checkpoints `best.pt`, `last.pt` y gráficos quedarán en `entrenamientos/<RUN_NAME>/`. Las métricas no se generan hasta que entrenes realmente.

## 1. Instalar librerías y comprobar GPU

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3,<9', 'roboflow', 'pyyaml'])
import torch, ultralytics
from importlib.metadata import version as package_version
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No hay GPU. Activa GPU en Colab y vuelve a ejecutar esta celda.')
print('GPU:', torch.cuda.get_device_name(0))

## 2. Conectar Drive y configurar el entrenamiento
Para una prueba corta usa otro `RUN_NAME` y `EPOCHS = 1`. Para reanudar una ejecución interrumpida conserva el nombre y los parámetros originales y cambia `RESUME = True`. Si quieres iniciar otra ejecución usa un nombre diferente. No borres checkpoints anteriores.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, yaml, shutil, os, math

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Sistema-Peaje')
for name in ('datasets', 'modelos', 'entrenamientos', 'evidencias'):
    (BASE / name).mkdir(parents=True, exist_ok=True)

WORKSPACE = 'franz-orozco'
PROJECT = 'vehicle-classification-v2-t9ukm'
VERSION = 1
SOURCE = 'roboflow'  # 'zip' si subiste la exportación YOLOv8 a Drive
ZIP_PATH = BASE / 'datasets' / f'm2_v{VERSION}.zip'
LOCAL = Path(f'/content/peaje_m2_v{VERSION}')
RUN_NAME = f'M2_segmentacion_v{VERSION}'
RUN = BASE / 'entrenamientos' / RUN_NAME
RESUME = False
CACHE_ZIP = True  # conserva una copia del dataset en Drive para nuevas sesiones
EPOCHS = 50
IMGSZ = 640
BATCH = 8  # reduce a 4 o 2 si no cabe en VRAM (nueva ejecución)
PATIENCE = 15
SEED = 42
CONF = 0.40  # umbral inicial de demostración, ajustar según métricas reales
EXPECTED = {'auto', 'bus', 'camion', 'minibus', 'moto'}
print('Resultados:', RUN)
print('Dataset de respaldo:', ZIP_PATH)

## 3. Descargar el dataset YOLOv8 de Roboflow (o recuperar la copia de Drive)
El notebook prefiere una copia ya guardada en Drive para no volver a descargar. Si cambias de versión, usa un ZIP y carpeta distintos. El ZIP debe proceder de la exportación de segmentación de tu proyecto, no de detección de objetos.

In [ ]:
import zipfile
from getpass import getpass

def extract_zip(src, target):
    target.mkdir(parents=True, exist_ok=True)
    root = target.resolve()
    with zipfile.ZipFile(src) as z:
        for item in z.infolist():
            p = (target / item.filename).resolve()
            if not p.is_relative_to(root):
                raise ValueError('Ruta insegura en ZIP')
        z.extractall(target)

if not LOCAL.exists() or not list(LOCAL.rglob('data.yaml')):
    if ZIP_PATH.is_file():
        extract_zip(ZIP_PATH, LOCAL)
    elif SOURCE == 'zip':
        raise FileNotFoundError(f'Sube el ZIP YOLOv8 a {ZIP_PATH}')
    elif SOURCE == 'roboflow':
        from roboflow import Roboflow
        from google.colab import userdata
        try:
            key = userdata.get('ROBOFLOW_API_KEY')
        except Exception:
            key = getpass('Clave privada vigente de Roboflow (no se mostrará): ')
        if not key:
            raise ValueError('Falta ROBOFLOW_API_KEY')
        rf = Roboflow(api_key=key)
        project = rf.workspace(WORKSPACE).project(PROJECT)
        dataset = project.version(VERSION).download('yolov8', location=str(LOCAL))
        del key
    else:
        raise ValueError('SOURCE debe ser roboflow o zip')

candidates = list(LOCAL.rglob('data.yaml'))
if len(candidates) != 1:
    raise RuntimeError(f'Se esperaba un solo data.yaml, se encontraron {len(candidates)}')
ORIGINAL_YAML = candidates[0]
ROOT = ORIGINAL_YAML.parent
print('Dataset preparado:', ROOT)

## 4. Validar clases, carpetas y polígonos
Se comprueba cada etiqueta antes de entrenar. Las filas YOLO-seg deben tener una clase y al menos tres pares de coordenadas. Los archivos vacíos pueden representar fondos negativos válidos. Si hay rectángulos de detección, coordenadas inválidas, clases incorrectas o un split sin etiquetas, se detiene el entrenamiento. Revisa visualmente las imágenes autoetiquetadas en Roboflow: una validación estructural no garantiza la calidad de las máscaras.

In [ ]:
from collections import Counter
from PIL import Image as PILImage

cfg = yaml.safe_load(ORIGINAL_YAML.read_text(encoding='utf-8'))
raw_names = cfg['names']
NAMES = {int(k): str(v) for k,v in raw_names.items()} if isinstance(raw_names, dict) else dict(enumerate(raw_names))
assert set(NAMES.values()) == EXPECTED and set(NAMES) == set(range(5)) and len(NAMES) == 5, f'Clases incompatibles: {NAMES}'
if 'nc' in cfg:
    assert int(cfg['nc']) == 5, 'nc debe ser 5'
print('Clases:', NAMES)
SPLITS = {'train':'train', 'val':'valid', 'test':'test'}
EXTS = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
IMAGES = {}
report = {}
errors = []
missing = []
for split, folder in SPLITS.items():
    image_dir = ROOT / folder / 'images'
    label_dir = ROOT / folder / 'labels'
    if split == 'val' and not image_dir.is_dir():
        image_dir, label_dir = ROOT/'val'/'images', ROOT/'val'/'labels'
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f'Falta images o labels en {split}: {image_dir}')
    images = sorted(p for p in image_dir.rglob('*') if p.suffix.lower() in EXTS)
    if not images:
        raise ValueError(f'{split} no tiene imágenes')
    counts = Counter()
    negatives = 0
    expected_labels = set()
    for img in images:
        lbl = label_dir / img.relative_to(image_dir).with_suffix('.txt')
        expected_labels.add(lbl)
        if not lbl.is_file():
            missing.append(str(img))
            negatives += 1
            continue
        lines = lbl.read_text(encoding='utf-8-sig').splitlines()
        lines = [line for line in lines if line.strip()]
        if not lines:
            negatives += 1
        for i, line in enumerate(lines, 1):
            vals = line.split()
            try:
                cls = int(vals[0])
                xy = list(map(float, vals[1:]))
                if cls not in NAMES or len(xy) < 6 or len(xy)%2 or not all(math.isfinite(v) and 0 <= v <= 1 for v in xy):
                    raise ValueError('clase o polígono inválido')
                points = list(zip(xy[::2], xy[1::2]))
                if len(set(points)) < 3:
                    raise ValueError('menos de tres vértices distintos')
                area2 = abs(sum(x*y2-x2*y for (x,y),(x2,y2) in zip(points,points[1:]+points[:1])))
                if area2 <= 1e-12:
                    raise ValueError('polígono sin área')
                counts[NAMES[cls]] += 1
            except (ValueError, IndexError) as e:
                errors.append(f'{lbl.name}:{i} {e}')
    orphan = set(label_dir.rglob('*.txt')) - expected_labels
    if orphan:
        print(f'AVISO: {split} tiene {len(orphan)} etiquetas sin imagen.')
    IMAGES[split] = images
    report[split] = {'imagenes':len(images),'negativas_o_sin_etiqueta':negatives,'instancias':{v:counts[v] for v in NAMES.values()}}
    if not sum(counts.values()):
        errors.append(f'{split}: sin objetos segmentados')
    if split == 'train' and any(counts[v]==0 for v in NAMES.values()):
        errors.append('Faltan clases en entrenamiento')
    if split != 'train' and any(counts[v]==0 for v in NAMES.values()):
        print(f'AVISO: {split} no contiene todas las clases; métricas por clase pueden no ser representativas.')
    SPLITS[split] = image_dir
print(json.dumps(report, ensure_ascii=False, indent=2))
if missing:
    print(f'AVISO: {len(missing)} imágenes sin TXT. YOLO puede tratarlas como fondo; revisa que no contengan vehículos sin etiquetar.')
    print('Ejemplos:', [Path(x).name for x in missing[:5]])
if errors:
    print('\n'.join(errors[:20]))
    raise RuntimeError(f'{len(errors)} problemas de anotación: corrige antes de entrenar.')

# Rutas absolutas estables en Colab; no modifica el YAML original.
DATA = Path(f'/content/m2_v{VERSION}_colab.yaml')
new_cfg = {'path':str(ROOT.resolve()), **{s:str(p.resolve()) for s,p in SPLITS.items()}, 'nc':5,'names':NAMES}
DATA.write_text(yaml.safe_dump(new_cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
(BASE/'evidencias'/f'M2_v{VERSION}_dataset.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('YAML listo:', DATA)

## 5. Comprobar duplicados exactos y visualizar ejemplos
Una misma captura o frames consecutivos no deberían aparecer en diferentes conjuntos. El hash detecta copias idénticas, no todas las variantes visuales; revisa el origen de los videos. Si aparecen duplicados, corrige la división en Roboflow antes de tomar las métricas como fiables.

In [ ]:
import hashlib, random
from collections import defaultdict
import matplotlib.pyplot as plt
from PIL import ImageDraw

hashes = defaultdict(set)
for split, images in IMAGES.items():
    for img in images:
        hashes[hashlib.sha256(img.read_bytes()).hexdigest()].add(split)
duplicados = sum(len(s)>1 for s in hashes.values())
print('Hashes idénticos entre conjuntos:', duplicados)
if duplicados:
    print('AVISO: revisa la fuga de datos antes de validar el rendimiento.')

rng = random.Random(SEED)
for img in rng.sample(IMAGES['train'], min(4,len(IMAGES['train']))):
    im = PILImage.open(img).convert('RGB')
    w,h = im.size
    draw = ImageDraw.Draw(im)
    label = SPLITS['train'].parent/'labels'/img.relative_to(SPLITS['train']).with_suffix('.txt')
    if label.exists():
        for line in label.read_text(encoding='utf-8-sig').splitlines():
            if not line.strip(): continue
            cls,*coords = line.split()
            xy = list(map(float,coords))
            pts = [(x*w,y*h) for x,y in zip(xy[::2],xy[1::2])]
            draw.line(pts+[pts[0]], fill='lime', width=max(2,w//300))
            draw.text(pts[0],NAMES[int(cls)],fill='yellow',stroke_width=1,stroke_fill='black')
    plt.figure(figsize=(9,6)); plt.imshow(im); plt.title(img.name[:75]); plt.axis('off'); plt.show()

if CACHE_ZIP and not ZIP_PATH.exists():
    archive = shutil.make_archive(str(LOCAL)+'_backup', 'zip', root_dir=ROOT)
    temp = ZIP_PATH.with_suffix('.zip.partial')
    shutil.copy2(archive,temp)
    os.replace(temp,ZIP_PATH)
    Path(archive).unlink()
    print('Copia del dataset guardada en Drive.')

## 6. Entrenar YOLOv8n-seg
Esta celda entrena con GPU. `best.pt` (mejor resultado de validación) y `last.pt` (último checkpoint) se escriben directamente en Drive en cada época guardada, además de checkpoints periódicos. No cierres Colab mientras se ejecuta. Para reanudar tras una desconexión, vuelve a ejecutar las celdas anteriores con `RESUME = True` y el mismo `RUN_NAME`. Si el entrenamiento ya terminó, utiliza la celda de evaluación en vez de reanudar. Si Drive se desconecta, comprueba los checkpoints antes de continuar.

In [ ]:
from ultralytics import YOLO
LAST = RUN/'weights'/'last.pt'
BEST = RUN/'weights'/'best.pt'
if RESUME:
    if not LAST.is_file():
        raise FileNotFoundError(f'No hay checkpoint para reanudar: {LAST}')
    model = YOLO(str(LAST))
    if model.task != 'segment':
        raise ValueError('Checkpoint incompatible')
    results = model.train(resume=True, data=str(DATA), device=0)
else:
    if RUN.exists() and any(RUN.iterdir()):
        raise FileExistsError('La ejecución ya existe. Cambia RUN_NAME o activa RESUME si se interrumpió.')
    model = YOLO('yolov8n-seg.pt')
    assert model.task == 'segment'
    results = model.train(
        data=str(DATA), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        device=0, workers=2, patience=PATIENCE, seed=SEED,
        pretrained=True, optimizer='auto', plots=True, save=True, save_period=5,
        cache=False, project=str(RUN.parent), name=RUN.name, exist_ok=True
    )
print('Checkpoints:', RUN/'weights')

## 7. Evaluar el mejor checkpoint sobre TEST
No utilices el conjunto de test para ajustar hiperparámetros repetidamente. Se registran métricas de máscaras y cajas; no son una garantía de rendimiento en un peaje real.

In [ ]:
from ultralytics import YOLO
BEST = RUN/'weights'/'best.pt'
if not BEST.is_file():
    raise FileNotFoundError(f'No se encontró {BEST}; termina el entrenamiento primero.')
modelo = YOLO(str(BEST))
assert modelo.task == 'segment'
model_names = {int(k):str(v) for k,v in modelo.names.items()} if isinstance(modelo.names,dict) else dict(enumerate(modelo.names))
assert model_names == NAMES, f'Orden de clases incompatible: {model_names}'
TEST_DIR = BASE/'evidencias'/f'{RUN_NAME}_test'
metricas = modelo.val(data=str(DATA), split='test', imgsz=IMGSZ, batch=BATCH, device=0, plots=True, project=str(TEST_DIR.parent), name=TEST_DIR.name, exist_ok=True)
metrics_dict = {str(k):float(v) for k,v in metricas.results_dict.items()}
metrics_dict.update({'mask_mAP50':float(metricas.seg.map50),'mask_mAP50_95':float(metricas.seg.map),'box_mAP50':float(metricas.box.map50),'box_mAP50_95':float(metricas.box.map)})
print(json.dumps(metrics_dict,indent=2))
(TEST_DIR/'metricas_test.json').write_text(json.dumps(metrics_dict,indent=2),encoding='utf-8')

## 8. Exportar los archivos para Antigravity
Ejecuta esta celda después de la evaluación. Si no puedes completarla, `best.pt` seguirá en la carpeta de entrenamiento. Los resultados de esta ejecución también se conservan allí. El archivo `vehiculos.pt` será una copia exacta del mejor checkpoint; no es necesario convertirlo a ONNX para el servicio Ultralytics actual.

In [ ]:
from datetime import datetime, timezone
import platform
from hashlib import sha256

DEST = BASE/'modelos'/'vehiculos.pt'
if DEST.exists() and sha256(DEST.read_bytes()).hexdigest() != sha256(BEST.read_bytes()).hexdigest():
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    shutil.copy2(DEST,DEST.with_name(f'vehiculos_anterior_{stamp}.pt'))
shutil.copy2(BEST, DEST)
manifest = {
    'modelo':'M2', 'task':'segment', 'weights':'vehiculos.pt',
    'sha256':sha256(DEST.read_bytes()).hexdigest(), 'base_model':'yolov8n-seg.pt',
    'classes':model_names, 'input_size':IMGSZ, 'confidence_threshold_initial':CONF,
    'dataset':{'workspace':WORKSPACE,'project':PROJECT,'version':VERSION,'format':'yolov8'},
    'training_run':RUN_NAME,'training_args_file':'args.yaml',
    'metrics_test':metrics_dict, 'environment':{'ultralytics':ultralytics.__version__,'torch':torch.__version__,'roboflow':package_version('roboflow'),'python':platform.python_version()},
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'note':'Umbral inicial no calibrado; validar en imágenes nuevas del peaje.'
}
manifest_path = BASE/'modelos'/'vehiculos.manifest.json'
manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('ENTREGAR AL PROYECTO:')
print(DEST)
print(manifest_path)
print('Respaldo y evidencias:', RUN, TEST_DIR)

## 9. Visualizar predicciones en imágenes de prueba
Se guardan hasta cinco predicciones del conjunto test en Drive. La máscara y el nombre de cada categoría deben verse correctamente. Después puedes ejecutar la celda opcional con una foto nueva autorizada, sin placas ni rostros identificables cuando no sean necesarios.

In [ ]:
from IPython.display import display, Image
PRED_DIR = BASE/'evidencias'/f'{RUN_NAME}_predicciones'
selected = random.Random(SEED).sample(IMAGES['test'],min(5,len(IMAGES['test'])))
preds = modelo.predict(source=[str(p) for p in selected], imgsz=IMGSZ, conf=CONF, device=0, save=True, project=str(PRED_DIR.parent), name=PRED_DIR.name, exist_ok=True)
for p in preds:
    display(PILImage.fromarray(p.plot()[:,:,::-1]))
print('Predicciones:', PRED_DIR)

### Prueba opcional con una fotografía nueva
Ejecuta esta celda manualmente solo si deseas subir una imagen desde tu computadora. Por defecto no solicita archivos al ejecutar todo el notebook.

In [ ]:
PROBAR_IMAGEN_PROPIA = False
if PROBAR_IMAGEN_PROPIA:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        for pred in modelo.predict(source=name,imgsz=IMGSZ,conf=CONF,device=0):
            display(PILImage.fromarray(pred.plot()[:,:,::-1]))

## 10. Evidencias y entrega
Ejecuta para ver gráficos disponibles. Los nombres de archivos pueden variar con la versión de Ultralytics.

In [ ]:
for folder in (RUN, TEST_DIR):
    print(folder)
    for name in ('results.png','confusion_matrix.png','confusion_matrix_normalized.png','MaskPR_curve.png','BoxPR_curve.png'):
        p=folder/name
        if p.exists():
            display(Image(filename=str(p),width=850))

### Qué archivo usar
| Archivo en Drive | Uso |
|---|---|
| `modelos/vehiculos.pt` | Copiar a `inferencia/modelos/vehiculos.pt` o almacenamiento de pesos en RunPod. Es el `best.pt` del entrenamiento. |
| `modelos/vehiculos.manifest.json` | Entregar a Antigravity para comprobar clases, esquema y versiones. |
| `entrenamientos/<RUN_NAME>/weights/last.pt` | Respaldo para reanudar si se interrumpe el entrenamiento. No es el modelo preferido para producción. |
| `entrenamientos/<RUN_NAME>/weights/best.pt` | Original del modelo elegido por validación. |
| `entrenamientos/<RUN_NAME>/results.csv`, `args.yaml`, gráficos | Evidencias y reproducibilidad. |
| `evidencias/<RUN_NAME>_test/metricas_test.json` | Métricas reales de test. |
| `datasets/m2_v1.zip` | Copia de recuperación si se habilitó el respaldo; no subir a GitHub. |

Configurar en la inferencia `M2_MODEL_PATH=modelos/vehiculos.pt`. Cargar con `YOLO(ruta)` y verificar `model.task == 'segment'`, los nombres del modelo y las máscaras `result.masks`. El archivo no resuelve OCR ni ejes (M1 y M3). Validar las predicciones reales antes de activar producción.

**Referencias:** [Roboflow Python SDK](https://roboflow.github.io/roboflow-python/), [Ultralytics: segmentación](https://docs.ultralytics.com/tasks/segment/), [formato de dataset](https://docs.ultralytics.com/datasets/segment/), [entrenamiento y reanudación](https://docs.ultralytics.com/modes/train/).